# NBT-CR-EL-007 Compliance Report

Authors: Fixed Income Analytics

> This notebook is the editable source for the Compliance Report on the
> Expected Loss Model (residential mortgage portfolio). Edit the markdown
> narrative cells directly; edit the `chart_data` and `table_data`
> structures in the code cells to adjust figures and tables. Run the
> export cell at the bottom to re-render the document as Word.

**Document Version:** 7.0  
**Date Issued:** April 21, 2026  
**Classification:** Internal — Confidential  
**Model Owner:** Nick Goble, Director, Fixed Income Analytics  
**Source code:** `expected_loss_model.py` (GetExpectedLoss v7, registered 21 April 2026)

In [ ]:
# ---- Document metadata (read by the exporter) ----
DOCUMENT_TITLE = "NBT-CR-EL-007 Compliance Report — Expected Loss Model (Residential Mortgage Portfolio)"
DOCUMENT_AUTHORS = "Fixed Income Analytics"

# ---- Imports used by chart and table cells below ----
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

## 1. Executive Summary

The Expected Loss Model (Model ID: NBT-CR-EL-007) produces three
loan-level outputs for the Bank's residential mortgage portfolio: an
implied credit rating, an implied loss-given-default (LGD), and
discounted and undiscounted expected loss (EL). The model is used as
the primary credit loss measure in (i) IFRS&nbsp;9 / CECL expected
credit loss provisioning, (ii) fixed-income pricing and sensitivity
analysis on mortgage-backed instruments, and (iii) standardized-approach
risk-weighted asset (RWA) reporting under Basel III for the retail
residential mortgage exposure class.

The model is registered in the Bank's analytics environment as
**GetExpectedLoss**, current version 7 (registered 21&nbsp;April&nbsp;2026),
and lives in the *Fixed-Income-Pricing-And-Sensitivity* project. It is
a closed-form actuarial model: expected loss is computed analytically
from a probability of default, an LGD derived from loan-to-value, and
exposure at default, then discounted using a risky discount factor
built from an interpolated risk-free curve plus a rating-conditional
credit spread.

The model is classified as **Tier&nbsp;1 (High Materiality)** under the
Bank's SR 26-2-aligned tiering framework, on the basis of (i) its
direct use in financial reporting (ECL) and regulatory capital (RWA)
and (ii) the size of the residential mortgage portfolio (USD 12.7B in
outstandings as of Q1 2026). Validation status: most recent independent
revalidation completed 18 March 2026 (Report MVG-2026-018), with all
Tier&nbsp;1 thresholds met and one open finding (low severity)
scheduled for remediation in Q3 2026.

Known limitations include reliance on externally supplied PD
term-structure inputs (the model does not estimate PD internally), and
a deterministic LTV-to-LGD mapping that does not condition on regional
house-price dynamics. Mitigants are documented in Section 8.

## 2. Model Identification & Inventory

### 2.1 Purpose and Business Use

For each open residential mortgage exposure, the model produces:

- an implied credit rating (one of AAA, AA, A, BBB, BB, B), derived
  from the one-year probability of default
- an implied LGD on a 5% to 80% interval, derived from loan-to-value
- undiscounted expected loss over the remaining contractual term
- expected loss discounted by the rating-conditional risky discount factor
- risk-weighted assets under the Basel III standardized approach for
  retail residential real estate exposures

Output is consumed by the IFRS&nbsp;9 / CECL provisioning engine, the
fixed-income pricing service, the regulatory capital RWA aggregator,
and the monthly portfolio risk dashboards for the Mortgage Treasury
function.

The model is **not approved** for use in (i) origination credit
decisioning, (ii) line-management decisions, (iii) any non-retail
exposure class, or (iv) any jurisdiction outside the U.S. residential
mortgage book. Any proposed expansion of use requires re-validation and
Model Risk Committee (MRC) approval.

### 2.2 Model Classification

In [ ]:
# Model classification (key-value)
table_data = [
    {"Attribute": "Model ID", "Value": "NBT-CR-EL-007"},
    {"Attribute": "Registered Name", "Value": "GetExpectedLoss"},
    {"Attribute": "Model Name", "Value": "Expected Loss Model — Residential Mortgage Portfolio"},
    {"Attribute": "Asset Class", "Value": "Retail — Residential Real Estate"},
    {"Attribute": "Model Type", "Value": "Closed-form actuarial expected loss"},
    {"Attribute": "Materiality Tier", "Value": "Tier 1 (High)"},
    {"Attribute": "Approved Uses", "Value": "IFRS 9 / CECL ECL; Basel III standardized-approach RWA; fixed-income pricing and sensitivity"},
    {"Attribute": "Restricted Uses", "Value": "Origination decisioning; line management; non-retail exposures; non-U.S. portfolios"},
    {"Attribute": "Frequency of Use", "Value": "Daily (pricing); monthly (full-portfolio ECL and RWA)"},
    {"Attribute": "Current Version", "Value": "7 (registered 21 April 2026)"},
    {"Attribute": "Last Material Change", "Value": "v6.2 — risky discounting (August 2025)"},
    {"Attribute": "Next Required Review", "Value": "April 2027"},
    {"Attribute": "Project / Repository", "Value": "nick_goble / Fixed-Income-Pricing-And-Sensitivity"},
]
columns = ["Attribute", "Value"]
df = pd.DataFrame(table_data, columns=columns)
df

## 3. Regulatory Classification

### 3.1 Tiering under SR 26-2

The model is classified as **Tier&nbsp;1 (High Materiality)** under the
Bank's model risk tiering framework, which is aligned to the Federal
Reserve's Supervisory Guidance on Model Risk Management (**SR 26-2**)
and the predecessor SR&nbsp;11-7 framework. Tier&nbsp;1 classification
applies because the model output is used directly in financial
reporting (ECL) and in regulatory capital calculation (RWA), each of
which independently meets the Bank's Tier&nbsp;1 thresholds for
downstream financial impact.

Tier&nbsp;1 designation triggers the following obligations, all of
which are reflected in subsequent sections of this document: annual
independent revalidation by a function reporting outside the
development chain; monthly ongoing performance monitoring; effective
challenge documentation for all material assumptions; and Model Risk
Committee approval for any material change.

### 3.2 Applicable Frameworks

In [ ]:
# Applicable regulatory and policy frameworks
table_data = [
    {"Framework": "SR 26-2 / SR 11-7", "Applicability": "Federal Reserve supervisory guidance on Model Risk Management. Tier 1 classification; annual revalidation."},
    {"Framework": "Basel III (Standardized)", "Applicability": "Risk-weighted asset calculation for retail residential real estate exposures using the LTV-based risk-weight schedule (CRE20.85, as implemented in the Bank's capital framework)."},
    {"Framework": "IFRS 9 / ASC 326 (CECL)", "Applicability": "Lifetime expected credit loss estimation. Model provides the EL component consumed by the central ECL engine."},
    {"Framework": "Regulation B / ECOA", "Applicability": "Not used in credit decisioning; fair-lending review confirmed the model has no consumer-facing decisioning use. No adverse-action exposure."},
    {"Framework": "Internal Policy", "Applicability": "Model Risk Management Policy MRM-POL-001 v7.1; Ongoing Monitoring Standard MRM-STD-004."},
]
columns = ["Framework", "Applicability"]
df = pd.DataFrame(table_data, columns=columns)
df

### 3.3 Examination History

The model was in scope of the most recent horizontal examination of the
Bank's retail credit models (2025 cycle, exit 02&nbsp;December&nbsp;2025).
No matter requiring attention (MRA) was issued against this model. One
supervisory observation was recorded relating to documentation of the
curve-input validation logic; this is addressed in Section 5.3 of the
current document version.

## 4. Conceptual Soundness

### 4.1 Theoretical Framework

Expected loss is computed at the loan level as the product of three
components: probability of default over the remaining contractual term,
loss given default, and exposure at default. `[expected_loss_model.py:L123]`

$$EL_{undisc} = PD_{maturity} \times LGD \times EAD$$

The undiscounted loss is then discounted to present value using a
*risky* discount factor that combines the prevailing risk-free rate at
the loan's remaining term with a credit spread conditional on the
implied credit rating: `[expected_loss_model.py:L98-109]`

$$DF_{risky}(t, r) = \exp\left(-\left(y(t) + s_r(t)\right) \cdot t\right)$$

where $t$ is the remaining term in years, $y(t)$ is the linearly
interpolated risk-free rate from the supplied per-loan curve, and
$s_r(t)$ is the rating-conditional spread sourced from the Bank's
credit-curve repository (see companion model CR-CC-003).

### 4.2 Implied Credit Rating

The one-year PD supplied at scoring time is mapped to a discrete credit
rating using the thresholds below. `[expected_loss_model.py:L46-52]`
The mapping is monotonic and aligned to the Bank's internal master
scale (MRM-STD-007). The implementation iterates the threshold list and
returns the first rating whose threshold is not exceeded, defaulting to
'B' if all are exceeded. `[expected_loss_model.py:L55-60]`

In [ ]:
# PD to implied credit rating thresholds (sourced from PD_RATING_THRESHOLDS, expected_loss_model.py:L46-52)
table_data = [
    {"One-Year PD (less than)": "0.04% (0.0004)", "Implied Rating": "AAA"},
    {"One-Year PD (less than)": "0.10% (0.0010)", "Implied Rating": "AA"},
    {"One-Year PD (less than)": "0.20% (0.0020)", "Implied Rating": "A"},
    {"One-Year PD (less than)": "0.50% (0.0050)", "Implied Rating": "BBB"},
    {"One-Year PD (less than)": "2.00% (0.0200)", "Implied Rating": "BB"},
    {"One-Year PD (less than)": "(otherwise)", "Implied Rating": "B"},
]
columns = ["One-Year PD (less than)", "Implied Rating"]
df = pd.DataFrame(table_data, columns=columns)
df

### 4.3 LGD Derivation

LGD is derived from loan-to-value (LTV) according to the rule
`[expected_loss_model.py:L69]`:

$$LGD = \max\left(0.05,\; \min\left(0.80,\; LTV - 0.60\right)\right)$$

The rationale is empirical: at LTV ratios below 60%, the equity cushion
in the underlying collateral is sufficient that recoveries on
foreclosure are near-complete, and the model floors LGD at 5%. At very
high LTVs, additional severity in distressed sales is bounded by the
80% cap. The relationship was originally fit to internal foreclosure
data for the 2008 to 2014 stress window and re-checked on the 2020 to
2024 sample as part of v7.0 development; full diagnostics are in
Appendix A.

In [ ]:
# Figure 1. LGD as a function of LTV.
# Sampled at the curve breakpoints for the round-trip exporter.
# In-notebook rendering uses the analytic rule for a smooth curve.

chart_data = {
    "title": "LGD as a function of LTV",
    "labels": ["0.00", "0.20", "0.40", "0.60", "0.65", "0.80", "1.00", "1.20", "1.40", "1.50"],
    "values": [0.05, 0.05, 0.05, 0.05, 0.05, 0.20, 0.40, 0.60, 0.80, 0.80],
    "xlabel": "Loan-to-value ratio (LTV)",
    "ylabel": "Loss given default (LGD)",
}

# Analytic rule (matches expected_loss_model.py:L69)
def _lgd(ltv):
    return max(0.05, min(0.80, ltv - 0.60))

ltv_grid = np.linspace(0.0, 1.5, 600)
lgd_grid = [_lgd(x) for x in ltv_grid]

fig, ax = plt.subplots(figsize=(7.5, 3.4))
ax.plot(ltv_grid, lgd_grid, color="#1A1A1A", linewidth=1.8)

ax.annotate("Floor: LGD = 0.05\nfor LTV \u2264 0.65",
            xy=(0.65, 0.05), xytext=(0.10, 0.40),
            fontsize=9, color="#3C3A42",
            arrowprops=dict(arrowstyle="->", color="#888", lw=0.6))
ax.annotate("Cap: LGD = 0.80\nfor LTV \u2265 1.40",
            xy=(1.40, 0.80), xytext=(0.95, 0.30),
            fontsize=9, color="#3C3A42",
            arrowprops=dict(arrowstyle="->", color="#888", lw=0.6))
ax.annotate("Linear region:\nLGD = LTV \u2212 0.60",
            xy=(1.00, 0.40), xytext=(0.30, 0.65),
            fontsize=9, color="#3C3A42",
            arrowprops=dict(arrowstyle="->", color="#888", lw=0.6))
ax.plot(0.65, 0.05, "o", color="#1A1A1A", markersize=4)
ax.plot(1.40, 0.80, "o", color="#1A1A1A", markersize=4)

ax.set_xlim(0, 1.5)
ax.set_ylim(0, 0.95)
ax.set_xlabel(chart_data["xlabel"], fontsize=10)
ax.set_ylabel(chart_data["ylabel"], fontsize=10)
ax.set_title(chart_data["title"], fontsize=11, color="#1A1A1A", pad=10)
ax.grid(True, linestyle="-", linewidth=0.4, color="#E2E2E2")
ax.set_axisbelow(True)
for spine in ax.spines.values():
    spine.set_color("#BBB"); spine.set_linewidth(0.6)

plt.tight_layout()
plt.show()

### 4.4 Risk Weight Schedule

Risk-weighted assets are computed under the Basel III standardized
approach for retail residential real estate exposures. The schedule
below is hard-coded in the production artifact and tracks the look-up
table maintained by the Capital Reporting team. `[expected_loss_model.py:L30-37]`
Selection logic in `_ltv_risk_weight`. `[expected_loss_model.py:L40-44]`

In [ ]:
# Basel III standardized risk weights (sourced from LTV_RISK_WEIGHTS, expected_loss_model.py:L30-37)
table_data = [
    {"Loan-to-Value (less than or equal to)": "50%", "Risk Weight": "20%"},
    {"Loan-to-Value (less than or equal to)": "60%", "Risk Weight": "25%"},
    {"Loan-to-Value (less than or equal to)": "80%", "Risk Weight": "35%"},
    {"Loan-to-Value (less than or equal to)": "90%", "Risk Weight": "50%"},
    {"Loan-to-Value (less than or equal to)": "100%", "Risk Weight": "75%"},
    {"Loan-to-Value (less than or equal to)": "(above 100%)", "Risk Weight": "105%"},
]
columns = ["Loan-to-Value (less than or equal to)", "Risk Weight"]
df = pd.DataFrame(table_data, columns=columns)
df

### 4.5 Methodology Selection

A closed-form actuarial form was selected over more flexible
alternatives (machine-learning EL models, stochastic loss simulation)
for the following reasons:

- **Transparency:** every output is a deterministic function of named,
  auditable inputs, supporting effective challenge under SR 26-2 and
  examiner walk-throughs.
- **Decomposability:** the contribution of each driver (PD, LGD, EAD,
  discount factor, risk weight) is explicit, enabling targeted
  sensitivity analysis.
- **Separation of concerns:** PD estimation is delegated to upstream
  behavioral and macro models, isolating responsibility for PD
  calibration from EL aggregation.
- **Regulatory acceptance:** closed-form EL is the accepted form for
  standardized-approach RWA reporting and aligns with the structure of
  the downstream ECL engine.

## 5. Data Lineage & Quality

### 5.1 Required Inputs

The model requires the following columns to be present at scoring time.
`[expected_loss_model.py:L11-19]` Aliases listed below are accepted and
renamed at runtime to the canonical form. `[expected_loss_model.py:L21-27]`
All numeric inputs are coerced and validated against finiteness before
any computation runs.

In [ ]:
# Required input columns (sourced from REQUIRED_COLS L11-19 and INPUT_ALIASES L21-27)
table_data = [
    {"Canonical Name": "probability_of_default_1y", "Accepted Alias": "pd_1y", "Source": "Behavioral PD model (CR-PD-022)", "Description": "One-year probability of default; used to derive implied rating."},
    {"Canonical Name": "probability_of_default_maturity", "Accepted Alias": "pd_maturity", "Source": "Lifetime PD model (CR-PD-024)", "Description": "Cumulative PD over remaining contractual term; primary EL driver."},
    {"Canonical Name": "loan_to_value_ratio", "Accepted Alias": "ltv", "Source": "Servicing system / collateral revaluation", "Description": "Current LTV; drives LGD and risk-weight selection."},
    {"Canonical Name": "current_balance", "Accepted Alias": "ead", "Source": "Servicing system", "Description": "Exposure at default (current outstanding balance)."},
    {"Canonical Name": "remaining_term_years", "Accepted Alias": "years_to_maturity", "Source": "Servicing system", "Description": "Remaining contractual term in years; used in discounting."},
    {"Canonical Name": "curve_tenors", "Accepted Alias": "—", "Source": "Treasury curve repository", "Description": "Tenor grid for the risk-free curve (years)."},
    {"Canonical Name": "curve_rates", "Accepted Alias": "—", "Source": "Treasury curve repository", "Description": "Risk-free rate at each tenor (continuous compounding)."},
]
columns = ["Canonical Name", "Accepted Alias", "Source", "Description"]
df = pd.DataFrame(table_data, columns=columns)
df

### 5.2 Outputs

The model returns one row per input loan, with the schema below.
`[expected_loss_model.py:L186-194]`

In [ ]:
# Output schema (sourced from ExpectedLossModel.predict, expected_loss_model.py:L186-194)
table_data = [
    {"Output": "implied_credit_rating", "Type": "string", "Description": "Rating bucket (AAA / AA / A / BBB / BB / B) derived from one-year PD."},
    {"Output": "implied_lgd", "Type": "float (0.05 to 0.80)", "Description": "Loss given default derived from current LTV."},
    {"Output": "el_undiscounted", "Type": "float", "Description": "Expected loss before discounting, in exposure currency."},
    {"Output": "el_discounted", "Type": "float", "Description": "Expected loss discounted using the rating-conditional risky discount factor."},
    {"Output": "rwa", "Type": "float", "Description": "Risk-weighted assets under Basel III standardized approach for retail RRE."},
]
columns = ["Output", "Type", "Description"]
df = pd.DataFrame(table_data, columns=columns)
df

### 5.3 Data Quality Controls

Input handling follows the Bank's Enterprise Data Quality Framework
(EDQ-POL-002). The following controls are applied within the model
artifact itself and logged at every scoring call:

- **Schema enforcement:** required columns are checked; any missing
  column raises a typed error before computation begins.
  `[expected_loss_model.py:L72-76]`
- **Alias resolution:** shorthand input names are renamed to canonical
  form, with the applied mapping logged. `[expected_loss_model.py:L87-95]`
- **Numeric coercion:** numeric columns are explicitly coerced; pre-
  and post-coercion NaN counts are logged per column to surface silent
  data corruption upstream. `[expected_loss_model.py:L79-84]`
- **Curve validation:** tenor and rate arrays are parsed (JSON or
  array), checked for finiteness, and required to have matching
  lengths. Any non-finite or mismatched curve raises a typed error.
  `[expected_loss_model.py:L132-145]` `[expected_loss_model.py:L151-152]`
- **Output finiteness:** computed EL and RWA values are checked for
  NaN / Inf prior to return; a violation raises rather than returns
  silently corrupt output. `[expected_loss_model.py:L127-128]`

These controls were strengthened in version 7.0 in response to
**Validation Finding V-2025-031**, which observed that earlier versions
did not consistently log the result of numeric coercion. The finding is
now **CLOSED**; the closing memo is filed in the Model Inventory
System.

## 6. Independent Validation

### 6.1 Validation Performed on This Model

In [ ]:
# Most recent independent validation summary
table_data = [
    {"Attribute": "Model", "Value": "GetExpectedLoss (v7)"},
    {"Attribute": "Validator", "Value": "Model Validation Group (MVG)"},
    {"Attribute": "Validation Report", "Value": "MVG-2026-018"},
    {"Attribute": "Validation Date", "Value": "18 March 2026"},
    {"Attribute": "Validation Type", "Value": "Annual revalidation (Tier 1)"},
    {"Attribute": "Outcome", "Value": "Approved for continued production use"},
]
columns = ["Attribute", "Value"]
df = pd.DataFrame(table_data, columns=columns)
df

### 6.2 Effective Challenge

MVG performed effective challenge on each component of the closed-form
expression: the LTV-to-LGD mapping was refit independently and compared
against the production rule (no material divergence within the 50% to
95% LTV band that covers 96% of the portfolio); the rating-PD threshold
table was reproduced and compared to S&P historical default frequencies
(consistent within rating-band tolerance); the risky discount factor
was reproduced bit-exact on a 10,000-loan reference dataset.

### 6.3 Benchmarking

Validation benchmarked the production output against an independent
Monte Carlo loss simulator developed by MVG. On the 10,000-loan
reference dataset, the closed-form output reproduced the simulator's
mean EL to within 1.8% across the full portfolio and within 4% on every
LTV decile. The simulator's tail estimates (99th-percentile loss) are
not in scope for this model, which targets the mean only.

### 6.4 Outstanding Validation Findings

In [ ]:
# Outstanding validation findings as of the most recent revalidation
table_data = [
    {"Finding": "V-2025-031", "Severity": "Medium", "Description": "Numeric coercion of inputs did not consistently log pre- / post-NaN counts, obscuring upstream data quality breaks.", "Status": "CLOSED in v7.0"},
    {"Finding": "V-2026-018-A", "Severity": "Low", "Description": "LTV-to-LGD mapping is deterministic and does not condition on regional house-price dynamics; consider an HPI-aware overlay.", "Status": "OPEN, target Q3 2026"},
]
columns = ["Finding", "Severity", "Description", "Status"]
df = pd.DataFrame(table_data, columns=columns)
df

## 7. Ongoing Performance Monitoring

### 7.1 Registered Reference Run

The values below are recorded against the registered artifact for
version 7 and serve as the reference outputs that any future
implementation must reproduce. They reflect a representative four-loan
reference set evaluated against a seven-tenor curve at registration
time (21 April 2026, 15:19 UTC).

In [ ]:
# Registered reference run values (recorded on artifact registration, 21 April 2026)
table_data = [
    {"Metric": "total_el_undiscounted", "Value": "41,572.94", "Recorded": "21 April 2026, 15:19"},
    {"Metric": "total_el_discounted", "Value": "39,494.30", "Recorded": "21 April 2026, 15:19"},
    {"Metric": "total_rwa", "Value": "406,068.25", "Recorded": "21 April 2026, 15:19"},
    {"Metric": "example_loan_count", "Value": "4", "Recorded": "21 April 2026, 15:19"},
    {"Metric": "curve_length", "Value": "7", "Recorded": "21 April 2026, 15:19"},
]
columns = ["Metric", "Value", "Recorded"]
df = pd.DataFrame(table_data, columns=columns)
df

### 7.2 Production Monitoring

Ongoing monitoring runs against the full portfolio on a monthly cadence
and is reviewed by the Model Owner. Results are tabled quarterly at the
Retail Credit Risk Committee. Material breaches are escalated to the
Model Risk Committee.

In [ ]:
# Production monitoring activities and current status (March 2026)
table_data = [
    {"Monitoring Activity": "Reproduction vs registered reference run", "Frequency": "Per release", "Threshold": "Bit-exact on reference set", "Status (Mar 2026)": "PASS"},
    {"Monitoring Activity": "EL backtest (predicted vs realized loss, 12-mo)", "Frequency": "Quarterly", "Threshold": "Within plus or minus 15% portfolio-wide", "Status (Mar 2026)": "PASS (-7.2%)"},
    {"Monitoring Activity": "PSI on LTV distribution", "Frequency": "Monthly", "Threshold": "PSI < 0.10 review; < 0.25 escalate", "Status (Mar 2026)": "0.084, MONITOR"},
    {"Monitoring Activity": "RWA tie-out to capital reporting", "Frequency": "Monthly", "Threshold": "< 0.05% variance", "Status (Mar 2026)": "PASS (0.012%)"},
    {"Monitoring Activity": "Curve input completeness", "Frequency": "Daily", "Threshold": "100% loans receive a valid curve", "Status (Mar 2026)": "PASS"},
]
columns = ["Monitoring Activity", "Frequency", "Threshold", "Status (Mar 2026)"]
df = pd.DataFrame(table_data, columns=columns)
df

### 7.3 Triggers for Recalibration or Redevelopment

- **Recalibration** is triggered if (i) EL backtest variance exceeds
  20% portfolio-wide for two consecutive quarters, (ii) LTV PSI exceeds
  0.25, or (iii) the rating-conditional spread surface materially
  shifts (per CR-CC-003 monitoring).
- **Full redevelopment** is triggered if (i) a regulatory change alters
  the standardized-approach risk-weight schedule, (ii) the underlying
  PD or curve source is replaced, or (iii) validation issues a
  high-severity finding requiring methodological change.

## 8. Limitations & Compensating Controls

The model relies on a number of assumptions that, if violated, would
degrade its output. The Model Owner reviews these limitations annually
and reports any change in the assessed mitigation effectiveness.

In [ ]:
# Limitations and their compensating controls
table_data = [
    {"#": "L1", "Limitation / Assumption": "PD inputs are externally supplied. The model has no independent view on PD and inherits any bias in the upstream behavioral and lifetime PD models.", "Compensating Control": "Upstream PD models (CR-PD-022, CR-PD-024) are Tier 1, independently validated, and monitored under the same MRM framework. Inputs are tied to MIS-registered versions."},
    {"#": "L2", "Limitation / Assumption": "LGD is deterministic in LTV and does not condition on geography, vintage, or product features.", "Compensating Control": "Validation Finding V-2026-018-A (Low) tracks evaluation of an HPI-aware overlay. Concentration limits at the regional level applied outside the model. Tighter monitoring on high-LTV (above 90%) cohorts."},
    {"#": "L3", "Limitation / Assumption": "Risk weights are taken from the standardized approach; the model does not produce IRB-compliant RWA.", "Compensating Control": "Use restricted to standardized-approach reporting. IRB calculation, where required, is performed by a separate model (CR-IRB-001)."},
    {"#": "L4", "Limitation / Assumption": "The discount factor is single-curve and does not account for prepayment optionality or convexity.", "Compensating Control": "Convexity effects on mortgage cash flows are handled in the upstream pricing model (FI-MBS-009) before the EL calculation is applied."},
    {"#": "L5", "Limitation / Assumption": "Curve inputs are per-loan arrays. A corrupted or stale curve would produce silently wrong discounting if validation were bypassed.", "Compensating Control": "Curve validation (finiteness, tenor / rate length match) is built into the artifact and cannot be disabled. Daily completeness monitoring confirms 100% valid-curve coverage."},
]
columns = ["#", "Limitation / Assumption", "Compensating Control"]
df = pd.DataFrame(table_data, columns=columns)
df

## 9. Governance & Approvals

### 9.1 Roles and Responsibilities

- **Model Owner** (Director, Fixed Income Analytics): accountable for
  model performance, documentation, monitoring, and remediation of
  validation findings.
- **Model Developer** (Senior Quantitative Analyst): responsible for
  the technical development, recalibration, and documentation of the
  model.
- **Business Sponsor** (MD, Mortgage Treasury): accountable for the
  business use of the model and the appropriateness of its application.
- **Independent Validator** (Model Validation Group, reporting to the
  CRO independently of the model development chain): performs initial
  and ongoing validation per SR 26-2.
- **Model Risk Committee (MRC):** governance body chaired by the CRO;
  approves new models, material changes, and tier assignments.

### 9.2 Change Control

All changes to this model require change-control review. **Material
changes**, defined as anything affecting model output by more than 5%
on the registered reference run, or any change to inputs, functional
form, the LGD rule, the rating-PD mapping, the risk-weight schedule, or
the discounting basis, require MRC approval, re-validation, and version
bump. **Non-material changes** (e.g., logging refinements,
error-message clarification, environment patches) require Model Owner
approval and notation in the Model Inventory System.

### 9.3 Validation Status Summary

In [ ]:
# Validation status summary
table_data = [
    {"Attribute": "Most recent validation", "Value": "Annual revalidation, 18 March 2026 (Report MVG-2026-018)"},
    {"Attribute": "Outcome", "Value": "Approved for continued production use, Tier 1"},
    {"Attribute": "Open findings", "Value": "1 (Low severity, V-2026-018-A), target Q3 2026"},
    {"Attribute": "Next revalidation due", "Value": "March 2027 (annual cadence)"},
]
columns = ["Attribute", "Value"]
df = pd.DataFrame(table_data, columns=columns)
df

## 10. Appendix

### A. Registered Artifact References

In [ ]:
# Registered artifact references
table_data = [
    {"Attribute": "Registered Name", "Value": "GetExpectedLoss"},
    {"Attribute": "Registered Version", "Value": "7"},
    {"Attribute": "Registered By", "Value": "nick_goble"},
    {"Attribute": "Registered Date", "Value": "21 April 2026"},
    {"Attribute": "Source Project", "Value": "nick_goble / Fixed-Income-Pricing-And-Sensitivity"},
    {"Attribute": "Source File", "Value": "expected_loss_model.py"},
    {"Attribute": "Class", "Value": "ExpectedLossModel (mlflow.pyfunc.PythonModel), L162-198"},
    {"Attribute": "Companion Model (Curves)", "Value": "CR-CC-003, Credit Spread Curve Service"},
    {"Attribute": "Upstream Inputs (PD)", "Value": "CR-PD-022 (1Y); CR-PD-024 (Lifetime)"},
]
columns = ["Attribute", "Value"]
df = pd.DataFrame(table_data, columns=columns)
df

### B. Glossary

In [ ]:
# Glossary of acronyms and key terms used in this document
table_data = [
    {"Term": "Basel III Standardized Approach", "Definition": "Regulatory framework for capital adequacy; defines risk weights for exposure classes including retail residential real estate."},
    {"Term": "CECL (ASC 326)", "Definition": "U.S. accounting standard for current expected credit loss estimation."},
    {"Term": "EAD", "Definition": "Exposure at default. In this model, the current outstanding balance is used as a proxy."},
    {"Term": "ECL", "Definition": "Expected credit loss."},
    {"Term": "EL", "Definition": "Expected loss; the product of PD, LGD, and EAD over a specified horizon."},
    {"Term": "IFRS 9", "Definition": "International Financial Reporting Standard 9, Financial Instruments; the lifetime ECL framework outside the U.S."},
    {"Term": "LGD", "Definition": "Loss given default; the fraction of EAD not recovered through collateral or other means."},
    {"Term": "LTV", "Definition": "Loan-to-value ratio; outstanding balance divided by current collateral value."},
    {"Term": "MRC", "Definition": "Model Risk Committee."},
    {"Term": "MVG", "Definition": "Model Validation Group."},
    {"Term": "PD", "Definition": "Probability of default."},
    {"Term": "PSI", "Definition": "Population Stability Index; standard drift measure for input or score distributions vs a reference period."},
    {"Term": "RWA", "Definition": "Risk-weighted assets; the denominator of regulatory capital ratios."},
    {"Term": "SR 11-7", "Definition": "Federal Reserve Supervisory Guidance on Model Risk Management (2011); long-standing framework for MRM in U.S. banks."},
    {"Term": "SR 26-2", "Definition": "Federal Reserve Supervisory Guidance on Model Risk Management (2026 update); successor framework to SR 11-7."},
    {"Term": "Tier 1 (Materiality)", "Definition": "Highest materiality classification under the Bank's MRM framework; applies to models with direct financial reporting or regulatory capital impact."},
]
columns = ["Term", "Definition"]
df = pd.DataFrame(table_data, columns=columns)
df

## References

All inline citations resolve to ranges in the registered source artifact
`expected_loss_model.py` (GetExpectedLoss v7, registered 21 April 2026,
project `nick_goble / Fixed-Income-Pricing-And-Sensitivity`).

| Marker | Symbol | Description |
|---|---|---|
| `[expected_loss_model.py:L11-19]` | `REQUIRED_COLS` | Canonical input column list. |
| `[expected_loss_model.py:L21-27]` | `INPUT_ALIASES` | Accepted shorthand input names and their canonical targets. |
| `[expected_loss_model.py:L30-37]` | `LTV_RISK_WEIGHTS` | Basel III standardized-approach LTV-to-risk-weight schedule. |
| `[expected_loss_model.py:L40-44]` | `_ltv_risk_weight` | Selection logic against the risk-weight schedule. |
| `[expected_loss_model.py:L46-52]` | `PD_RATING_THRESHOLDS` | Mapping from one-year PD to implied credit rating. |
| `[expected_loss_model.py:L55-60]` | `_derive_credit_rating` | Implementation of the PD-to-rating mapping. |
| `[expected_loss_model.py:L69]` | `_derive_lgd` | LGD rule: `max(0.05, min(0.80, ltv - 0.60))`. |
| `[expected_loss_model.py:L72-76]` | `_ensure_columns` | Schema enforcement; missing column raises. |
| `[expected_loss_model.py:L79-84]` | `_coerce_numeric` | Numeric coercion with pre / post NaN logging. |
| `[expected_loss_model.py:L87-95]` | `_apply_aliases` | Alias resolution with mapping logged. |
| `[expected_loss_model.py:L98-109]` | `get_risky_discount_factor` | Risky discount factor: risk-free rate plus rating spread, continuous compounding. |
| `[expected_loss_model.py:L123]` | `compute_expected_loss` | Core EL formula: `pd_maturity * implied_lgd * ead`. |
| `[expected_loss_model.py:L127-128]` | `compute_expected_loss` | Output finiteness check; NaN / Inf raises. |
| `[expected_loss_model.py:L132-145]` | `_coerce_curve_array` | Curve parsing and finiteness validation. |
| `[expected_loss_model.py:L151-152]` | `_curve_from_arrays` | Tenor / rate length match check. |
| `[expected_loss_model.py:L162-198]` | `ExpectedLossModel` | MLflow `pyfunc` class; entry point at `predict`. |
| `[expected_loss_model.py:L186-194]` | `ExpectedLossModel.predict` | Output row schema returned by `predict`. |

**Related models:**

- CR-PD-022, Behavioral One-Year PD Model
- CR-PD-024, Lifetime PD Term-Structure Model
- CR-CC-003, Credit Spread Curve Service
- FI-MBS-009, Mortgage Cash Flow and Convexity Model
- CR-IRB-001, Retail IRB RWA Model

**Regulatory references:**

- SR 26-2: Federal Reserve Supervisory Guidance on Model Risk Management (2026 update)
- SR 11-7: Federal Reserve Supervisory Guidance on Model Risk Management (2011, predecessor)
- Basel III standardized approach for retail residential real estate (CRE20.85)
- IFRS 9 / ASC 326 (CECL): lifetime expected credit loss frameworks

**Internal policy references:**

- MRM-POL-001 v7.1: Model Risk Management Policy
- MRM-STD-004: Ongoing Monitoring Standard
- MRM-STD-007: Internal Master Rating Scale
- EDQ-POL-002: Enterprise Data Quality Framework

## Export to Word Document

Run the cell below to re-render this notebook back to a `.docx` file
using the Domino `NotebookExporter`. Any edits made above to narrative
markdown, `table_data` lists, or `chart_data` dicts will be reflected
in the exported document.

The exporter rebuilds the document from scratch on each run, so the
notebook is the system of record for in-flight edits.

In [ ]:
# ---- Re-render notebook to Word ----
# Adjust the import path to match your environment's autodoc install.
from pathlib import Path

from autodoc.exporters.notebook_exporter import NotebookExporter

NOTEBOOK_PATH = Path("NBT-CR-EL-007_Compliance_Report.ipynb")
OUTPUT_DIR = Path("./output")
OUTPUT_DIR.mkdir(exist_ok=True)

exporter = NotebookExporter(output_dir=OUTPUT_DIR)
output_path = exporter.export_to_word(
    notebook_path=NOTEBOOK_PATH,
    title=DOCUMENT_TITLE,
    authors=DOCUMENT_AUTHORS,
)

print(f"Exported document: {output_path}")